In [4]:
# Importa bibliotecas necessárias
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [5]:
# Definindo os caminho da nova pasta no PC pessoal
# O caminho usa '../../' pois o notebook está em scripts/Redução de Dimensionalidade/
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste = 'cicids2017_teste.csv'

nome_dados_treinamento_reduzidos = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv'
nome_dados_teste_reduzidos = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv'

In [6]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")

display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1979513, 81) | Teste: (848363, 81)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.006760,0.855604,0.352941,3.229166e-05,0.000009,0.000003,2.945736e-06,9.153974e-09,0.001531,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.625956,0.006760,0.352941,2.272266e-03,0.000009,0.000017,0.000000e+00,4.576987e-09,0.000000,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.006760,0.848402,0.352941,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.801816,0.000809,1.000000,1.233333e-06,0.000005,0.000007,6.821705e-06,1.830795e-07,0.001773,0.018925,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.583032,0.000809,1.000000,1.449567e-03,0.000005,0.000007,4.496124e-06,3.509023e-07,0.001168,0.012473,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [7]:
# 2. Removendo características identificadoras/enviesadas antes da seleção por MDI
# No dataset tratado atual, o campo identificador/enviesado ainda presente é a porta de destino.
# A coluna duplicada de Fwd Header Length também é removida para reduzir redundância.
nomes_enviesados = {
    'Flow ID',
    'Source IP',
    'Source Port',
    'Destination IP',
    'Destination Port',
    'Protocol',
    'Timestamp',
    'Fwd Header Length.1'
}

colunas_para_remover = []
fwd_header_length_vistos = 0

for coluna in df_treino.columns:
    nome_normalizado = coluna.strip().lower()

    if coluna in nomes_enviesados:
        colunas_para_remover.append(coluna)

    if nome_normalizado == 'fwd header length':
        fwd_header_length_vistos += 1
        if fwd_header_length_vistos > 1:
            colunas_para_remover.append(coluna)

    if nome_normalizado.startswith('fwd header length.'):
        colunas_para_remover.append(coluna)

colunas_para_remover = [coluna for coluna in dict.fromkeys(colunas_para_remover) if coluna != 'Label']
colunas_treino_existentes = [coluna for coluna in colunas_para_remover if coluna in df_treino.columns]
colunas_teste_existentes = [coluna for coluna in colunas_para_remover if coluna in df_teste.columns]

df_treino = df_treino.drop(columns=colunas_treino_existentes)
df_teste = df_teste.drop(columns=colunas_teste_existentes)

print('Características removidas para mitigar overfitting:', colunas_treino_existentes)
print(f'Dimensões após remoção - Treino: {df_treino.shape} | Teste: {df_teste.shape}')

# 3. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

Características removidas para mitigar overfitting: ['Source Port', 'Destination Port', 'Protocol', 'Fwd Header Length.1']
Dimensões após remoção - Treino: (1979513, 77) | Teste: (848363, 77)


In [8]:
# 4. Aplicando Seleção de Características MDI (Mean Decrease in Impurity) com Random Forest
print("\nIniciando o treinamento do Random Forest para extração de features (MDI)...")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()

print(f"Treinamento RF concluído! Tempo extração: {(fim_fs - inicio_fs):.4f} segundos.")

importancias = rf_fs.feature_importances_
features = X_treino.columns

df_importancias = pd.DataFrame({'Feature': features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

print("\nQuantidade total de features:", len(features))

# Selecionando as 51 features mais importantes (conforme sugerido na literatura por Neto e Gomes)
top_51_features = df_importancias.head(51)['Feature'].tolist()

quantidade_de_features_descartadas = len(features) - len(top_51_features)
print("Quantidade de features selecionadas (top 51):", len(top_51_features))
print("Quantidade de features descartadas:", quantidade_de_features_descartadas)

# Mostra qual foi o corte de importância para as features selecionadas
importancia_corte = df_importancias[df_importancias['Feature'] == top_51_features[-1]]['Importancia'].values[0]
print(f"Importância mínima para entrar no top 51: {importancia_corte:.6f}")
display(df_importancias.tail(quantidade_de_features_descartadas + 5))

print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))


Iniciando o treinamento do Random Forest para extração de features (MDI)...
Treinamento RF concluído! Tempo extração: 66.3956 segundos.

Quantidade total de features: 76
Quantidade de features selecionadas (top 51): 51
Quantidade de features descartadas: 25
Importância mínima para entrar no top 51: 0.002621


,Feature,Importancia
27,Bwd IAT Max,3.438520e-03
68,Active Mean,2.920115e-03
25,Bwd IAT Mean,2.740683e-03
71,Active Min,2.706856e-03
24,Bwd IAT Total,2.620871e-03
18,Flow IAT Min,2.527920e-03
4,Total Length of Bwd Packets,2.514892e-03
26,Bwd IAT Std,2.154481e-03
42,FIN Flag Count,1.945748e-03
28,Bwd IAT Min,1.732957e-03



Top 10 features mais importantes:


,Feature,Importancia
11,Bwd Packet Length Mean,0.055883
51,Average Packet Size,0.053185
39,Packet Length Mean,0.051992
12,Bwd Packet Length Std,0.048513
41,Packet Length Variance,0.044265
64,Init_Win_bytes_forward,0.043264
40,Packet Length Std,0.042768
9,Bwd Packet Length Max,0.040962
3,Total Length of Fwd Packets,0.037278
53,Avg Bwd Segment Size,0.035490


In [9]:
# 5. Reduzindo a dimensionalidade dos conjuntos de treino e teste
X_treino_reduzido = X_treino[top_51_features]
X_teste_reduzido = X_teste[top_51_features]

print(f"Novas dimensões - Treino: {X_treino_reduzido.shape} | Teste: {X_teste_reduzido.shape}")

# Salvando os datasets reduzidos em CSV para uso posterior
X_treino_reduzido.to_csv(caminho_pasta_tratado + nome_dados_treinamento_reduzidos, index=False)
print(f"Dataset de treino reduzido salvo em: {caminho_pasta_tratado + nome_dados_treinamento_reduzidos}")

X_teste_reduzido.to_csv(caminho_pasta_tratado + nome_dados_teste_reduzidos, index=False)
print(f"Dataset de teste reduzido salvo em: {caminho_pasta_tratado + nome_dados_teste_reduzidos}")


Novas dimensões - Treino: (1979513, 51) | Teste: (848363, 51)
Dataset de treino reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv
Dataset de teste reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv


In [10]:
# 6. Convertendo as Labels de Texto para Números
print("\nCodificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_)


Codificando as labels para a Rede Neural...


In [11]:
# 7. Construindo a Arquitetura da Rede Neural Profunda (DNN) com a entrada reduzida
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino_reduzido.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [12]:
# 8. O Treinamento (Medindo o Custo Computacional)
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino_reduzido, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...
Epoch 1/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9454 - loss: 0.1670 - val_accuracy: 0.9634 - val_loss: 0.0930
Epoch 2/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9653 - loss: 0.0896 - val_accuracy: 0.9689 - val_loss: 0.0752
Epoch 3/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9678 - loss: 0.0786 - val_accuracy: 0.9710 - val_loss: 0.0670
Epoch 4/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9697 - loss: 0.0731 - val_accuracy: 0.9748 - val_loss: 0.0636
Epoch 5/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9714 - loss: 0.0694 - val_accuracy: 0.9738 - val_loss: 0.0607
Epoch 6/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9728 - loss: 0.0664 - val_accuracy: 0.9794 - val_loss: 0.0576
Epoch 7/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9749 - loss: 0.0624 - val_accuracy: 0.9778 - val_loss: 0.0533
Ep

In [13]:
# 9. Classificação dos Dados de Teste
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste_reduzido)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

# 10. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
26512/26512 ━━━━━━━━━━━━━━━━━━━━ 12s 466us/step
Classificação concluída! Tempo de previsão: 19.4095 segundos.

=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===


,Pred_BENIGN,Pred_Bot,Pred_DDoS,Pred_DoS GoldenEye,Pred_DoS Hulk,Pred_DoS Slowhttptest,Pred_DoS slowloris,Pred_FTP-Patator,Pred_Heartbleed,Pred_Infiltration,Pred_PortScan,Pred_SSH-Patator,Pred_Web Attack – Brute Force,Pred_Web Attack – Sql Injection,Pred_Web Attack – XSS
Real_BENIGN,671450,4,94,63,3238,220,50,6,0,0,5889,0,0,0,0
Real_Bot,564,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Real_DDoS,238,0,38060,4,5,0,0,0,0,0,0,0,0,0,0
Real_DoS GoldenEye,64,0,0,3095,0,2,2,0,0,0,0,0,0,0,0
Real_DoS Hulk,729,0,0,0,68538,0,0,0,0,0,0,0,0,0,0
Real_DoS Slowhttptest,19,0,0,0,0,1627,12,0,0,0,0,0,0,0,0
Real_DoS slowloris,7,0,0,1,0,22,1727,0,0,0,0,0,0,0,0
Real_FTP-Patator,67,0,0,0,0,0,4,2255,0,0,0,0,0,0,0
Real_Heartbleed,5,0,0,0,0,0,0,0,1,0,0,0,0,0,0
Real_Infiltration,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.99      0.99    681014
                       Bot       0.00      0.00      0.00       564
                      DDoS       1.00      0.99      1.00     38307
             DoS GoldenEye       0.98      0.98      0.98      3163
                  DoS Hulk       0.95      0.99      0.97     69267
          DoS Slowhttptest       0.87      0.98      0.92      1658
             DoS slowloris       0.96      0.98      0.97      1757
               FTP-Patator       1.00      0.97      0.98      2326
                Heartbleed       1.00      0.17      0.29         6
              Infiltration       0.00      0.00      0.00        10
                  PortScan       0.89      0.96      0.92     47836
               SSH-Patator       1.00      0.49      0.66      1821
  Web Attack – Brute Force       1.00      0.05      0.

In [14]:
# 7. Exportacao das tabelas em formato LaTeX
from IPython.display import Markdown, display

CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'Bot': 'Bot', 'DDoS': 'DDoS', 'DoS GoldenEye': 'DoS GE', 'DoS Hulk': 'Hulk', 'DoS Slowhttptest': 'Slowhttp', 'DoS slowloris': 'Slowloris', 'FTP-Patator': 'FTP', 'Heartbleed': 'Heart', 'Infiltration': 'Infil', 'PortScan': 'PortScan', 'SSH-Patator': 'SSH', 'Web Attack - Brute Force': 'Brute', 'Web Attack - Sql Injection': 'SQL', 'Web Attack - XSS': 'XSS', 'Desconhecido': 'Desconh'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acur?cia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{M?dia Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{M?dia Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precis?o", "Revoca??o", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_matriz_confusao = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confus?o - Random Forest",
    "table:rf_completo_mc",
)
latex_relatorio_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relat?rio de M?tricas do Modelo Random Forest",
    "table:rf_completo_metricas",
)

print(latex_matriz_confusao)
print("\n")
print(latex_relatorio_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrrr}
            \hline
                                             & BENIGN      & Bot     & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & Infil  & PortScan   & SSH      & Web Attack – Brute Force & Web Attack – Sql Injection & Web Attack – XSS \\
            \hline
            Real\_BENIGN                     & \ok{671450} & \err{4} & \err{94}   & \err{63}  & \err{3238} & \err{220} & \err{50}  & \err{6}   & 0      & 0      & \err{5889} & 0        & 0                        & 0                          & 0                \\
            Real\_Bot                        & \err{564}   & \ok{0}  & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0      & 0          & 0        & 0                        & 0                          & 0                \\
            Real\_DDoS                       & \err